In [6]:
from glob import glob
from collections import defaultdict
from openpyxl import load_workbook
import smtplib
from email.mime.text import MIMEText
from email.utils import formatdate
from datetime import datetime

# =========================
# ① 各店舗の注文合計を集計
# =========================

# 商品ごとの注文合計
total_orders = defaultdict(int)

# order_で始まるファイルを自動取得
files = glob("order_*.xlsx")

print("対象ファイル")
for file in files:
    print(file)

# 各ファイル処理
for file in files:

    wb = load_workbook(file, data_only=True)
    ws = wb.active

    # 1行目 = 商品名
    products = [cell.value for cell in ws[1]]

    # 2行目 = 注文数
    quantities = [cell.value for cell in ws[2]]

    # 商品ごとに合計
    for product, qty in zip(products, quantities):

        # 商品名なしはスキップ
        if product is None:
            continue

        # 数量なしは0扱い
        if qty is None:
            qty = 0

        total_orders[product] += qty

# =========================
# ② inventory.xlsx 読み込み
# =========================

inventory_wb = load_workbook("inventory.xlsx", data_only=True)
inventory_ws = inventory_wb.active

# 最終行取得（最新在庫）
last_row = inventory_ws.max_row

# C列以降の商品在庫取得
inventory_stock = {}

col = 3  # C列

while True:

    product_name = inventory_ws.cell(row=1, column=col).value

    # 商品名が空なら終了
    if product_name is None:
        break

    stock_qty = inventory_ws.cell(row=last_row, column=col).value

    if stock_qty is None:
        stock_qty = 0

    inventory_stock[product_name] = stock_qty

    col += 1

# =========================
# ③ pickup.xlsx 読み込み
# =========================

pickup_wb = load_workbook("pickup.xlsx", data_only=True)
pickup_ws = pickup_wb.active

pickup_threshold = {}
pickup_order_qty = {}

col = 2  # B列

while True:

    product_name = pickup_ws.cell(row=1, column=col).value

    # 商品名なしなら終了
    if product_name is None:
        break

    # 閾値
    threshold = pickup_ws.cell(row=2, column=col).value

    # 発注数量
    order_qty = pickup_ws.cell(row=3, column=col).value

    if threshold is None:
        threshold = 0

    if order_qty is None:
        order_qty = 0

    pickup_threshold[product_name] = threshold
    pickup_order_qty[product_name] = order_qty

    col += 1

# =========================
# ④ 在庫計算
# =========================

print("\n=== 商品別 合計注文数 ===")

for product, total_qty in total_orders.items():
    print(f"{product}: {total_qty}")

# =========================
# ⑤ 発注必要商品判定
# =========================

pickup_list = []

print("\n=== 在庫チェック結果 ===")

for product, current_stock in inventory_stock.items():

    # 注文数
    ordered_qty = total_orders.get(product, 0)

    # 出荷後在庫
    remaining_stock = current_stock - ordered_qty

    # 閾値
    threshold = pickup_threshold.get(product)

    # 発注数量
    pickup_qty = pickup_order_qty.get(product)

    # pickup.xlsxに存在しない商品はスキップ
    if threshold is None:
        continue

    print(
        f"{product} | 現在在庫:{current_stock} "
        f"| 注文数:{ordered_qty} "
        f"| 残在庫:{remaining_stock}"
    )

    # 閾値未満なら発注対象
    if remaining_stock < threshold:

        pickup_list.append({
            "product": product,
            "qty": pickup_qty
        })

# =========================
# ⑥ メール本文作成
# =========================

mail_body = ""

mail_body += "関係各位\n\n"
mail_body += "いつも大変お世話になっております。\n"
mail_body += "新たに以下商品の注文をお願いできますでしょうか。\n\n"

# 発注対象商品
if len(pickup_list) > 0:

    for item in pickup_list:

        mail_body += f"品名：{item['product']}\n"
        mail_body += f"注文数：{item['qty']}\n\n"

else:

    mail_body += "現在、追加発注が必要な商品はありません。\n"

# =========================
# ⑦ メール内容表示
# =========================

print("\n")
print("===================================")
print("メール文面")
print("===================================")
print("")

print(mail_body)

# =========================
# ⑧ Gmail送信設定
# =========================

gmail_account = "i.ishii.1982.336@gmail.com"
gmail_password = "mbab cesc igdb pyem"

to_email = "i.ishii.1982.336@gmail.com"

# 日付取得
today = datetime.now().strftime("%Y年%m月%d日")

subject = f"野菜発注 {today}"

# =========================
# ⑨ メール作成
# =========================

msg = MIMEText(mail_body, "plain", "utf-8")

msg["Subject"] = subject
msg["From"] = gmail_account
msg["To"] = to_email
msg["Date"] = formatdate()

# =========================
# ⑩ Gmail送信
# =========================

try:

    smtp = smtplib.SMTP("smtp.gmail.com", 587)

    smtp.starttls()

    smtp.login(gmail_account, gmail_password)

    smtp.send_message(msg)

    smtp.quit()

    print("Gmail送信完了")

except Exception as e:

    print("メール送信エラー")
    print(e)

対象ファイル
order_D_20230524.xlsx
order_A_20230524.xlsx
order_C_20230524.xlsx
order_B_20230524.xlsx

=== 商品別 合計注文数 ===
トマト: 31.0
キャベツ: 21.0
レタス: 42.0
ほうれん草: 23.0
ニンジン: 32.0
白菜: 25.0
大根: 15.0

=== 在庫チェック結果 ===
トマト | 現在在庫:91.0 | 注文数:31.0 | 残在庫:60.0
キャベツ | 現在在庫:73.0 | 注文数:21.0 | 残在庫:52.0
レタス | 現在在庫:103.0 | 注文数:42.0 | 残在庫:61.0
白菜 | 現在在庫:84.0 | 注文数:25.0 | 残在庫:59.0
ほうれん草 | 現在在庫:75.0 | 注文数:23.0 | 残在庫:52.0
大根 | 現在在庫:48.0 | 注文数:15.0 | 残在庫:33.0
ニンジン | 現在在庫:50.0 | 注文数:32.0 | 残在庫:18.0


メール文面

関係各位

いつも大変お世話になっております。
新たに以下商品の注文をお願いできますでしょうか。

品名：ニンジン
注文数：80.0


Gmail送信完了
